# Training LLM on a free Google Colab instance - Autotrain

Welcome to this notebook that will show you how to finetune a LLM model using your own dataset. We'll be using:
- the recent peft library and bitsandbytes for loading large models in 4-bit.
- autotrain to run the training

The fine-tuning method will rely on a recent method called "Low Rank Adapters" (LoRA), instead of fine-tuning the entire model you just have to fine-tune these adapters and load them properly inside the model. After fine-tuning the model you can also share your adapters on the 🤗 Hub and load them very easily. Let's get started!

Note that this could be used for any model that supports device_map (i.e. loading the model with accelerate).

## Step 0 - Define some helper functions :
- Enable text wrapping so we don't have to scroll horizontally
- Define a wrapper function which pass our query to the model for inference and return decoded model's completion(response).

In [ ]:
from IPython.display import HTML, display

def set_css():
  display(HTML('''
  <style>
    pre {
        white-space: pre-wrap;
    }
  </style>
  '''))

get_ipython().events.register('pre_run_cell', set_css)

In [ ]:
def get_completion(query: str, model, tokenizer) -> str:
  device = "cuda:0"

  prompt_template = """
  Below is an instruction that describes a task. Write a response that appropriately completes the request.
  ### Question:
  {query}

  ### Answer:
  """
  prompt = prompt_template.format(query=query)

  encodeds = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)

  model_inputs = encodeds.to(device)


  generated_ids = model.generate(**model_inputs, max_new_tokens=1000, do_sample=True, pad_token_id=tokenizer.eos_token_id)
  decoded = tokenizer.batch_decode(generated_ids)
  return (decoded[0])

## Setup Runtime
For fine-tuning Mistral 7b, a GPU instance is essential. Follow the directions below:
1. Go to Runtime (located in the top menu bar).
2. Select Change Runtime Type.
3. Choose T4 GPU (or a comparable option).

## Step 1 - Installing necessary packages and logging to Hugging Face

First, install the dependencies below to get started. As these features are available on the main branches only, we need to install the libraries below from source.

In [ ]:
!pip install -q pandas
!pip install -q autotrain-advanced safetensors
!autotrain setup --update-torch

**Connect to Hugging Face for model upload**

Logging to Hugging Face
To make sure the model can be uploaded to be used for Inference, it's necessary to log in to the Hugging Face hub.

Getting a Hugging Face token
Steps:

Navigate to this URL: https://huggingface.co/settings/tokens
Create a write token and copy it to your clipboard
Run the code below and enter your token

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 2 - Load the dataset

Now we'll load and prepare our dataset using the repository's data loading utilities.

In [ ]:
# Clone the repository if running in Colab
import os
if not os.path.exists('src'):
    !git clone https://github.com/backup-bdg-5/datasets.git
    %cd datasets

# Install required dependencies
!pip install -q -r requirements.txt

In [ ]:
# Import repository modules
import sys
import yaml
import pandas as pd
from datasets import load_dataset

# Add repository to path if needed
if not os.path.exists('src/data/loaders.py'):
    sys.path.append(os.path.abspath('.'))

# Load the repository's data utilities
from src.data.loaders import DatasetLoader
from src.data.preprocessors import TextPreprocessor
from src.utils.tokenization import get_tokenizer

In [ ]:
# Load configuration
config_path = "configs/config.yaml"
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Initialize dataset loader
dataset_loader = DatasetLoader(config_path)

In [ ]:
# Try to load a dataset from the repository's configuration
try:
    # Try loading one of the instruction datasets from config
    dataset_name = "HuggingFaceH4/instruction-dataset"
    dataset = dataset_loader.load_dataset(dataset_name, split="train")
    print(f"Loaded dataset {dataset_name} with {len(dataset)} examples")
except Exception as e:
    print(f"Error loading dataset from repository config: {e}")
    print("Falling back to finance dataset from the original notebook")
    # Fallback to the finance dataset from the original notebook
    dataset = load_dataset("ronal999/finance-alpaca-demo", split='train')
    # Take a subset for demonstration
    dataset = dataset.shard(num_shards=6, index=0)
    print(f"Loaded finance dataset with {len(dataset)} examples")

In [ ]:
# Display a few examples from the dataset
df = dataset.to_pandas()
df.head(5)

In [ ]:
# Preprocess the dataset using the repository's text preprocessor
text_preprocessor = TextPreprocessor(config)

# Format the dataset for instruction tuning
def format_instruction_dataset(example):
    # Adapt this function based on the dataset structure
    if 'instruction' in example and 'output' in example:
        instruction = text_preprocessor.clean_text(example['instruction'])
        response = text_preprocessor.clean_text(example['output'])
    elif 'instruction' in example and 'response' in example:
        instruction = text_preprocessor.clean_text(example['instruction'])
        response = text_preprocessor.clean_text(example['response'])
    elif 'text' in example:
        # For datasets with just text
        return {"prompt": text_preprocessor.clean_text(example['text'])}
    else:
        # Default format - use the first field as instruction and second as response
        keys = list(example.keys())
        instruction = text_preprocessor.clean_text(example[keys[0]])
        response = text_preprocessor.clean_text(example[keys[1]]) if len(keys) > 1 else ""
    
    # Create the prompt format
    prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.
### Question:
{instruction}

### Answer:
{response}"""
    
    return {"prompt": prompt}

In [ ]:
# Apply the formatting function
formatted_dataset = dataset.map(format_instruction_dataset)

# Save the formatted dataset to CSV for AutoTrain
formatted_dataset.to_csv("train.csv")
print(f"Saved formatted dataset to train.csv with {len(formatted_dataset)} examples")

## Step 3 - Meet AutoTrain and run the training !

## Overview of AutoTrain command

#### Short overview of what the command flags do.

- `!autotrain`: Command executed in environments like a Jupyter notebook to run shell commands directly. `autotrain` is an automatic training utility.

- `llm`: A sub-command or argument specifying the type of task

- `--train`: Initiates the training process.

- `--project_name`: Sets the name of the project

- `--model`: Specifies original model that is hosted on Hugging Face.
change model if you wish, you can use most of the text-generation models from Hugging Face Hub

- `--data_path .`: The path to the dataset for training. The "." refers to the current directory. The `train.csv` file needs to be located in this directory.

- `--use_peft`: Use of Parameter-Efficient-Finetuning to reduce the use of memory

- `--use_int4`: Use of INT4 quantization to reduce model size and speed up inference times at the cost of some precision.

- `--learning_rate 2e-4`: Sets the learning rate for training to 0.0002.

- `--train_batch_size 4`: Sets the batch size for training to 4.

- `--num_train_epochs 3`: The training process will iterate over the dataset 3 times.

In [ ]:
# Set up model parameters
import subprocess
from datetime import datetime

# Get model parameters from config
model_size = config['model']['size']
model_config = config['model']['sizes'][model_size]

# Set the base model - using Mistral 7B as default
MODEL_NAME = "mistralai/Mistral-7B-v0.1"

# Set project name
PROJECT_NAME = f"finetuned-model-{datetime.now().strftime('%Y%m%d')}"

# Get your Hugging Face username
try:
    result = subprocess.run(["huggingface-cli", "whoami"], capture_output=True, text=True)
    username = result.stdout.strip()
    if not username:
        username = input("Enter your Hugging Face username: ")
except:
    username = input("Enter your Hugging Face username: ")

# Set the repository ID for uploading the model
REPO_ID = f"{username}/{PROJECT_NAME}"

print(f"Base model: {MODEL_NAME}")
print(f"Project name: {PROJECT_NAME}")
print(f"Model will be uploaded to: {REPO_ID}")

In [ ]:
# Get training parameters from config
training_config = config['training']
instruction_tuning = next(stage for stage in training_config['stages'] if stage['name'] == 'instruction_tuning')

# Extract learning rate and other parameters
learning_rate = instruction_tuning['learning_rate']['initial']
epochs = instruction_tuning['epochs']
batch_size = config['data_processing']['batching']['train_batch_size']

# Adjust batch size for Colab environment
colab_batch_size = 1  # Safer for Colab's limited memory

print(f"Learning rate: {learning_rate}")
print(f"Epochs: {epochs}")
print(f"Original batch size: {batch_size}, adjusted for Colab: {colab_batch_size}")

In [ ]:
# Run AutoTrain for fine-tuning
!autotrain llm \
    --train \
    --project_name {PROJECT_NAME} \
    --model {MODEL_NAME} \
    --data_path . \
    --text-column prompt \
    --learning_rate {learning_rate} \
    --train_batch_size {colab_batch_size} \
    --num_train_epochs {epochs} \
    --trainer sft \
    --use_peft \
    --use_int4 \
    --fp16 \
    --lora-r 16 \
    --lora-alpha 32 \
    --lora-dropout 0.05 \
    --target_modules q_proj,v_proj \
    --push_to_hub \
    --repo_id {REPO_ID}

## Step 4 - Completed! Load the model for inference

Now we'll load the fine-tuned model and test it.

In [ ]:
import torch
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load the model configuration
peft_model_id = REPO_ID
config = PeftConfig.from_pretrained(peft_model_id)

# Load the base model with 4-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path, 
    return_dict=True, 
    load_in_4bit=True, 
    device_map='auto'
)

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path)

# Load the LoRA adapter
model = PeftModel.from_pretrained(model, peft_model_id)
print(f"Successfully loaded the model {peft_model_id} into memory")

In [ ]:
# Test the model with some examples
test_queries = [
    "Explain the concept of neural networks in simple terms.",
    "What are the key differences between supervised and unsupervised learning?",
    "How can I implement a basic recommendation system?"
]

for query in test_queries:
    print(f"\n\nQuery: {query}")
    print("\nResponse:")
    result = get_completion(query=query, model=model, tokenizer=tokenizer)
    print(result)

## Step 5 - Integration with Repository Structure

Now let's integrate the fine-tuned model with the repository's deployment structure.

In [ ]:
# Create output directories
import os
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = os.path.join("outputs", f"run_{timestamp}")
os.makedirs(output_dir, exist_ok=True)

# Create subdirectories
subdirs = [
    "data",
    "checkpoints",
    "tokenizer",
    "logs",
    "evaluation",
    "flask_deployment",
    "coreml_models"
]

for subdir in subdirs:
    os.makedirs(os.path.join(output_dir, subdir), exist_ok=True)

print(f"Created output directories in {output_dir}")

In [ ]:
# Save model metadata for deployment
import json

model_metadata = {
    "model_name": PROJECT_NAME,
    "base_model": MODEL_NAME,
    "adapter_model": peft_model_id,
    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "training_parameters": {
        "learning_rate": learning_rate,
        "epochs": epochs,
        "batch_size": colab_batch_size,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "target_modules": ["q_proj", "v_proj"],
        "quantization": "int4"
    },
    "dataset": {
        "size": len(formatted_dataset),
        "format": "instruction-tuning"
    }
}

# Save metadata
metadata_path = os.path.join(output_dir, "model_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(model_metadata, f, indent=2)

print(f"Saved model metadata to {metadata_path}")

In [ ]:
# Create a deployment configuration
deployment_config = {
    "model": {
        "base_model": MODEL_NAME,
        "adapter_model": peft_model_id,
        "quantization": "int4",
        "max_length": 4096,
        "temperature": 0.7,
        "top_p": 0.9,
        "top_k": 50
    },
    "deployment": {
        "type": "flask",
        "batch_size": 1,
        "max_concurrent_requests": 5,
        "timeout": 60
    }
}

# Save the configuration
config_path = os.path.join(output_dir, "deployment_config.json")
with open(config_path, "w") as f:
    json.dump(deployment_config, f, indent=2)

print(f"Saved deployment configuration to {config_path}")

## Step 6 - Deploy the Model using Repository's Flask Server

Now we'll use the repository's Flask deployment utilities to create a server for the model.

In [ ]:
# Import the repository's Flask export module
from src.deployment.flask_export import FlaskExporter

# Create a Flask exporter
flask_exporter = FlaskExporter(
    model_path=peft_model_id,
    output_dir=os.path.join(output_dir, "flask_deployment"),
    config=deployment_config
)

# Export the model for Flask deployment
flask_app_path = flask_exporter.export()
print(f"Exported Flask application to {flask_app_path}")

In [ ]:
# Create a simple test script for the Flask server
test_script = """
import requests
import json

# Server URL
url = "http://localhost:5000/generate"

# Test queries
test_queries = [
    "Explain the concept of neural networks in simple terms.",
    "What are the key differences between supervised and unsupervised learning?",
    "How can I implement a basic recommendation system?"
]

# Send requests
for query in test_queries:
    print(f"\nQuery: {query}")
    
    # Send request
    response = requests.post(url, json={"query": query})
    
    # Check if request was successful
    if response.status_code == 200:
        result = response.json()
        print(f"Response:\n{result['response']}")
    else:
        print(f"Error: {response.status_code}")
        print(response.text)
"""

# Save the test script
test_script_path = os.path.join(output_dir, "flask_deployment", "test_server.py")
with open(test_script_path, "w") as f:
    f.write(test_script)

print(f"Created test script at {test_script_path}")

## Step 7 - Evaluate the Model

Let's evaluate the model using the repository's evaluation utilities.

In [ ]:
# Import evaluation utilities
from src.utils.metrics import compute_metrics

# Create a small evaluation dataset
eval_data = [
    {
        "instruction": "Explain the concept of neural networks in simple terms.",
        "reference": "Neural networks are computing systems inspired by the human brain. They consist of interconnected nodes (neurons) that process and transmit information. Neural networks learn from examples to recognize patterns and make predictions without being explicitly programmed for specific tasks."
    },
    {
        "instruction": "What are the key differences between supervised and unsupervised learning?",
        "reference": "Supervised learning uses labeled data where the algorithm learns to map inputs to known outputs. Unsupervised learning works with unlabeled data to find patterns and relationships without predefined outputs. Supervised learning is used for classification and regression tasks, while unsupervised learning is used for clustering and dimensionality reduction."
    },
    {
        "instruction": "How can I implement a basic recommendation system?",
        "reference": "A basic recommendation system can be implemented using collaborative filtering, which analyzes user behavior and preferences to make recommendations. You can start by collecting user-item interaction data, creating a user-item matrix, calculating similarity between users or items, and then predicting ratings for unseen items based on similar users' preferences."
    }
]

# Generate predictions
predictions = []
for item in eval_data:
    query = item["instruction"]
    result = get_completion(query=query, model=model, tokenizer=tokenizer)
    
    # Extract just the response part
    response_parts = result.split("### Answer:")
    if len(response_parts) > 1:
        prediction = response_parts[1].strip()
    else:
        prediction = result
    
    predictions.append(prediction)

# Compute metrics
references = [item["reference"] for item in eval_data]
metrics = compute_metrics(predictions, references)

print("Evaluation Metrics:")
for metric_name, metric_value in metrics.items():
    print(f"{metric_name}: {metric_value}")

# Save evaluation results
eval_results = {
    "metrics": metrics,
    "examples": [
        {
            "instruction": item["instruction"],
            "reference": item["reference"],
            "prediction": pred
        }
        for item, pred in zip(eval_data, predictions)
    ]
}

eval_results_path = os.path.join(output_dir, "evaluation", "results.json")
with open(eval_results_path, "w") as f:
    json.dump(eval_results, f, indent=2)

print(f"\nSaved evaluation results to {eval_results_path}")

## Conclusion

In this notebook, we've demonstrated how to:

1. Set up the environment for fine-tuning
2. Load and prepare a dataset using the repository's utilities
3. Fine-tune a model using AutoTrain with LoRA
4. Load and test the fine-tuned model
5. Integrate the model with the repository's deployment structure
6. Create a Flask deployment server
7. Evaluate the model's performance

The fine-tuned model is now ready for deployment using the repository's deployment utilities.